# iREL NLP Task - Ready-to-Run Colab Notebook

This notebook sets up and runs the iREL Code-Mixed Pedagogical Flow pipeline from your GitHub repo.

Run cells in order from top to bottom.

In [14]:
# 1) Settings (edit only if needed)
REPO_URL = "https://github.com/AnishRacherla/irel-nlp-task.git"
PROJECT_DIR = "irel-nlp-task"
VIDEO_ID = "video_2"  # Try: video_1, video_2, video_3, video_4
RUN_ALL = False

In [15]:
print("hi")

hi


In [16]:
# 2) Clone repository
import os
import shutil

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

!git clone {REPO_URL}
%cd {PROJECT_DIR}

Cloning into 'irel-nlp-task'...
remote: Enumerating objects: 92, done.
remote: Total 92 (delta 0), reused 0 (delta 0), pack-reused 92 (from 2)
Receiving objects: 100% (92/92), 56.19 MiB | 47.51 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/irel-nlp-task/irel-nlp-task/irel-nlp-task


In [ ]:
# 3) Write config.yaml with API key directly
import os
os.chdir(f"/content/{PROJECT_DIR}")
os.makedirs("config", exist_ok=True)

config_content = """
video_sources:
  video_1:
    url: "https://www.youtube.com/watch?v=XV-lIaO00H8"
    language: auto
    domain: Computer Science
    duration_minutes: 10
  video_2:
    url: "https://www.youtube.com/watch?v=IlWB81vEH7g"
    language: auto
    domain: Computer Science
    duration_minutes: 10
  video_3:
    url: "https://www.youtube.com/watch?v=cOSTc6qBRQw"
    language: auto
    domain: Computer Science
    duration_minutes: 10
  video_4:
    url: "https://www.youtube.com/watch?v=SkE2kD2U4tU"
    language: auto
    domain: Magnetism
    duration_minutes: 30

transcription:
  model: whisper
  whisper_model_size: base
  language: auto
  output_format: json

language_processing:
  detect_code_mixing: true
  primary_languages: [en, hi, ta, te, kn]
  standardization_method: hybrid

concept_extraction:
  method: hybrid
  llm_provider: groq
  model: gpt-4o-mini
  groq_model: llama-3.3-70b-versatile
  min_concept_confidence: 0.7
  max_concepts_per_video: 20

prerequisite_mapping:
  method: hybrid
  confidence_threshold: 0.6

output:
  format: json

api_keys:
  groq_api_key: "YOUR_GROQ_API_KEY_HERE"
  openai_api_key: ""
"""

with open("config/config.yaml", "w") as f:
    f.write(config_content.strip())

print("config/config.yaml written successfully.")

In [ ]:
# 4) Check GPU — Runtime > Change runtime type > T4 GPU recommended
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("KeyBERT + sentence-transformers will use GPU automatically.")

GPU available: False


In [ ]:
# 5) Install dependencies
import os
os.chdir(f"/content/{PROJECT_DIR}")   # ensure correct directory regardless of cell order

!apt-get update -qq
!apt-get install -y ffmpeg -qq
!pip -q install -U pip
!pip -q install -r requirements.txt
!pip -q install groq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 118 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 77.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# 6) Install local package + NLP assets
import os; os.chdir(f"/content/{PROJECT_DIR}")
!pip -q install -e .
!python -m spacy download en_core_web_sm

import nltk
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("averaged_perceptron_tagger")

from src.pipeline import PedagogicalFlowPipeline
print("Environment is ready.")

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pedagogical-flow-extractor (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 129.0 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


Environment is ready.


In [ ]:
# 7) Show configured video IDs from config.yaml
import yaml

with open("config/config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

video_sources = cfg.get("video_sources", {})
print("Configured videos:", list(video_sources.keys()))
for vid, meta in video_sources.items():
    print(f"- {vid}: {meta.get('url', '')}")
print("LLM provider: Groq /", cfg.get("concept_extraction", {}).get("groq_model", "n/a"))

In [ ]:
# 8) Run pipeline
import subprocess

if RUN_ALL:
    cmd = ["python", "main.py", "--process-all"]
else:
    cmd = ["python", "example_usage.py", "--video-id", VIDEO_ID]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Pipeline run failed. Check logs above.")

In [ ]:
# 9) Preview generated JSON output
import json
import os

json_path = f"outputs/graphs/{VIDEO_ID}_complete_output.json"
if not os.path.exists(json_path):
    # fallback if RUN_ALL or different output path from script
    candidates = [p for p in os.listdir("outputs/graphs") if p.endswith("_complete_output.json")]
    if not candidates:
        raise FileNotFoundError("No *_complete_output.json found in outputs/graphs")
    json_path = os.path.join("outputs/graphs", candidates[0])

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

concepts = data.get("concepts", [])
relationships = data.get("relationships", [])
print("Output file:", json_path)
print("Concept count:", len(concepts))
print("Relationship count:", len(relationships))
print("First 5 concepts:")
for c in concepts[:5]:
    print("-", c.get("name", c.get("id", "unknown")))

In [ ]:
# 10) Display interactive HTML visualization in notebook
import os
from IPython.display import IFrame, display

html_path = f"outputs/visualizations/{VIDEO_ID}_interactive_graph.html"
if not os.path.exists(html_path):
    html_candidates = [p for p in os.listdir("outputs/visualizations") if p.endswith("_interactive_graph.html")]
    if not html_candidates:
        raise FileNotFoundError("No *_interactive_graph.html found in outputs/visualizations")
    html_path = os.path.join("outputs/visualizations", html_candidates[0])

print("Showing:", html_path)
display(IFrame(src=html_path, width=1100, height=650))

In [ ]:
# 11) Download output files to your machine
from google.colab import files
import zipfile

zip_name = "irel_outputs.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ["outputs/graphs", "outputs/visualizations"]:
        for root, _, filenames in os.walk(folder):
            for name in filenames:
                path = os.path.join(root, name)
                zf.write(path)

files.download(zip_name)

## Notes
- If a run fails, re-run the install cells and then restart runtime.
- If processing is slow, switch Colab runtime to GPU.
- Set RUN_ALL = True to process all configured videos.